# 03 — Qiskit vs PennyLane vs Cirq: Compilation Comparison

This notebook compares how different quantum frameworks compile the same algorithm. This is the **Quantum Framework Compilation Benchmark**.

**Key idea:** The same quantum algorithm compiled through different frameworks produces different circuit metrics. WestQuant Open provides a common comparison layer.

In [ ]:
# Install if needed
# !pip install westquant[qiskit] pennylane cirq

from qiskit import QuantumCircuit
from qiskit.circuit.library import QFTGate
from westquant_qiskit import circuit_metrics, QiskitAdapter
import pandas as pd
import matplotlib.pyplot as plt

print("WestQuant Open — Framework Compilation Benchmark")
print("=" * 50)

## Step 1: Build the Same Algorithm in Qiskit

We use QFT as the reference algorithm since it's available in Qiskit's circuit library.

In [ ]:
# Build QFT circuits at different sizes using Qiskit
qiskit_circuits = {}
for n in [4, 6, 8, 10, 12]:
    qc = QuantumCircuit(n)
    qc.append(QFTGate(n), range(n))
    qc = qc.decompose(reps=3)
    qiskit_circuits[n] = qc
    m = circuit_metrics(qc)
    print(f"Qiskit QFT-{n}: depth={m['depth']}, 2q={m['two_qubit_gates']}, size={m['size']}")

print(f"\nBuilt {len(qiskit_circuits)} Qiskit QFT circuits")

## Step 2: Build the Same Algorithm in PennyLane

In [ ]:
try:
    import pennylane as qml
    pennylane_available = True
    print(f"PennyLane version: {qml.__version__}")
except ImportError:
    pennylane_available = False
    print("PennyLane not installed. Install with: pip install pennylane")
    print("Skipping PennyLane comparison (showing Qiskit-only results below).")

if pennylane_available:
    # Build QFT in PennyLane
    def make_pennylane_qft(n):
        """Build QFT using PennyLane operations."""
        wires = list(range(n))
        for i in range(n):
            qml.Hadamard(wires=i)
            for j in range(i + 1, n):
                angle = 3.14159 / 2 ** (j - i)
                qml.ControlledPhaseShift(angle, wires=[i, j])
        # Bit reversal (swap qubits)
        for i in range(n // 2):
            qml.SWAP(wires=[i, n - 1 - i])
        return wires

    # Get the tape (circuit) for each size
    pennylane_circuits = {}
    for n in [4, 6, 8, 10, 12]:
        dev = qml.device("default.qubit", wires=n)
        @qml.qnode(dev)
        def circuit():
            make_pennylane_qft(n)
            return qml.state()
        circuit()
        tape = circuit.tape if hasattr(circuit, 'tape') else None
        pennylane_circuits[n] = tape
        if tape:
            ops = tape.operations
            depth = qml.drawer.depth(ops) if hasattr(qml.drawer, 'depth') else len(ops)
            two_q = sum(1 for op in ops if len(op.wires) == 2)
            print(f"PennyLane QFT-{n}: depth={depth}, 2q={two_q}, ops={len(ops)}")
else:
    pennylane_circuits = {}

## Step 3: Compare Frameworks

Using WestQuant's framework-neutral metrics, we can compare circuits from different frameworks on the same scale.

In [ ]:
# Collect metrics from all frameworks into a comparison table
comparison = []

for n in [4, 6, 8, 10, 12]:
    # Qiskit
    m = circuit_metrics(qiskit_circuits[n])
    comparison.append({
        "framework": "Qiskit",
        "algorithm": "QFT",
        "qubits": n,
        "depth": m["depth"],
        "two_qubit_gates": m["two_qubit_gates"],
        "size": m["size"],
        "swap_gates": m["swap_gates"],
    })

    # PennyLane (if available)
    if pennylane_available and n in pennylane_circuits and pennylane_circuits[n]:
        tape = pennylane_circuits[n]
        ops = tape.operations
        depth = qml.drawer.depth(ops) if hasattr(qml.drawer, 'depth') else len(ops)
        two_q = sum(1 for op in ops if len(op.wires) == 2)
        comparison.append({
            "framework": "PennyLane",
            "algorithm": "QFT",
            "qubits": n,
            "depth": depth,
            "two_qubit_gates": two_q,
            "size": len(ops),
            "swap_gates": sum(1 for op in ops if op.name == "SWAP"),
        })

df = pd.DataFrame(comparison)
print("=== Framework Compilation Benchmark ===")
print(df.to_string(index=False))

In [ ]:
# Plot the comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

frameworks = df["framework"].unique()
colors = {"Qiskit": "steelblue", "PennyLane": "coral", "Cirq": "green"}

for ax, metric, title in zip(axes, ["depth", "two_qubit_gates", "size"],
                               ["Depth", "2-Qubit Gates", "Total Gates"]):
    for fw in frameworks:
        sub = df[df["framework"] == fw]
        ax.plot(sub["qubits"], sub[metric], "o-", label=fw, color=colors.get(fw, "gray"), linewidth=2, markersize=8)
    ax.set_xlabel("Qubits")
    ax.set_ylabel(title)
    ax.set_title(f"QFT: {title} by Framework")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle("Quantum Framework Compilation Benchmark — QFT", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("framework_benchmark.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 4: Import into WestQuant's Neutral Representation

WestQuant's `QiskitAdapter` converts framework-specific circuits into a framework-neutral `Representation` object. This is the first step toward cross-framework comparison and ML dataset generation.

In [ ]:
# Import Qiskit circuits into WestQuant's neutral representation
adapter = QiskitAdapter()

print("=== WestQuant Neutral Representation ===")
for n in [4, 6, 8]:
    rep = adapter.import_native(qiskit_circuits[n])
    print(f"\nQFT-{n}:")
    print(f"  ID:      {rep.id}")
    print(f"  Kind:    {rep.kind.name}")
    print(f"  Framework: {rep.framework}")
    print(f"  Metrics: {rep.payload['metrics']}")
    print(f"  Instructions: {len(rep.payload.get('instructions', []))}")

# Save comparison table
df.to_csv("framework_comparison.csv", index=False)
print("\nSaved: framework_comparison.csv")

## Summary

This benchmark can become a recurring publication: **Quantum Framework Compilation Benchmark**. Same circuits, same targets, different frameworks — measured on a common scale.

### Next steps
- Add Cirq to the comparison
- Run with noise models (see `05_hardware_targeting.ipynb`)
- Generate ML training data across frameworks (see `02_generate_ml_dataset.ipynb`)